## Libraries

In [14]:
# weather data 
import meteostat as ms

# from src
from src.spark_config import initialize_spark 
from src.helper_functions import get_paths, get_coordinates 

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract, current_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType, ArrayType
)

# other libraries
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime, timedelta
from pathlib import Path

# specify java version
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Initialize

In [15]:
# set spark session
spark = initialize_spark()

Spark Version: 4.3.0
Application ID: local-1795182736944
Spark UI: http://10.255.255.254:4040


## Paths

In [16]:
STATSBOMB_PATHS = get_paths()

In [17]:
competitions_schema = StructType([
        StructField("competition_gender", StringType()),
        StructField("competition_id", LongType()), 
        StructField("competition_international", BooleanType()),
        StructField("competition_name", StringType()),
        StructField("competition_youth", BooleanType()),
        StructField("country_name", StringType()),
        StructField("match_available", StringType()),
        StructField("match_available_360", StringType()),
        StructField("match_updated", StringType()),
        StructField("match_updated_360", StringType()),
        StructField("season_id", LongType()),
        StructField("season_name", StringType())
    ])
competitions_df = spark.read.schema(competitions_schema).json(STATSBOMB_PATHS["COMPETITIONS"], multiLine=True)
competitions_df = competitions_df.withColumn("loaded_at", current_timestamp())

In [18]:
matches_schema = StructType([
        StructField("away_score", LongType()),
        StructField("away_team", StructType([
            StructField("away_team_gender", StringType()),
            StructField("away_team_group", StringType()),
            StructField("away_team_id", LongType()),
            StructField("away_team_name", StringType()),
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("managers", ArrayType(StructType([
                StructField("country", StructType([
                    StructField("id", LongType()),
                    StructField("name", StringType()),
                ])),
                StructField("dob", StringType()),
                StructField("id", LongType()),
                StructField("name", StringType()),
                StructField("nickname", StringType()),
            ]))),
        ])),
        StructField("competition", StructType([
            StructField("competition_id", LongType()),
            StructField("competition_name", StringType()),
            StructField("country_name", StringType()),
        ])),
        StructField("competition_stage", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("home_score", LongType()),
        StructField("home_team", StructType([
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType())
            ])),
            StructField("home_team_gender", StringType()),
            StructField("home_team_group", StringType()),
            StructField("home_team_id", LongType()),
            StructField("home_team_name", StringType()),
            StructField("managers", ArrayType(StructType([
                StructField("country", StructType([
                    StructField("id", LongType()),
                    StructField("name", StringType()),
                ])),
                StructField("dob", StringType()),
                StructField("id", LongType()),
                StructField("name", StringType()),
                StructField("nickname", StringType()),
            ]))),
        ])),
        StructField("kick_off", StringType()),
        StructField("last_updated", StringType()),
        StructField("last_updated_360", StringType()),
        StructField("match_date", StringType()),
        StructField("match_id", LongType()),
        StructField("match_status", StringType()),
        StructField("match_status_360", StringType()),
        StructField("match_week", LongType()),
        StructField("metadata", StructType([
            StructField("data_version", StringType()),
            StructField("shot_fidelity_version", StringType()),
            StructField("xy_fidelity_version", StringType()),
        ])),
        StructField("referee", StructType([
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("season", StructType([
            StructField("season_id", LongType()),
            StructField("season_name", StringType()),
        ])),
        StructField("stadium", StructType([
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])
matches_df = spark.read.schema(matches_schema).json(STATSBOMB_PATHS["MATCHES"], multiLine=True, recursiveFileLookup=True)
matches_df = matches_df.withColumn("loaded_at", current_timestamp())

In [19]:
print(matches_df)

DataFrame[away_score: bigint, away_team: struct<away_team_gender:string,away_team_group:string,away_team_id:bigint,away_team_name:string,country:struct<id:bigint,name:string>,managers:array<struct<country:struct<id:bigint,name:string>,dob:string,id:bigint,name:string,nickname:string>>>, competition: struct<competition_id:bigint,competition_name:string,country_name:string>, competition_stage: struct<id:bigint,name:string>, home_score: bigint, home_team: struct<country:struct<id:bigint,name:string>,home_team_gender:string,home_team_group:string,home_team_id:bigint,home_team_name:string,managers:array<struct<country:struct<id:bigint,name:string>,dob:string,id:bigint,name:string,nickname:string>>>, kick_off: string, last_updated: string, last_updated_360: string, match_date: string, match_id: bigint, match_status: string, match_status_360: string, match_week: bigint, metadata: struct<data_version:string,shot_fidelity_version:string,xy_fidelity_version:string>, referee: struct<country:struc

In [20]:
events_schema = StructType([
    StructField("50_50", StructType([
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("ball_receipt", StructType([
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("ball_recovery", StructType([
        StructField("recovery_failure", BooleanType()),
    ])),
    StructField("block", StructType([
        StructField("deflection", BooleanType()),
        StructField("offensive", BooleanType()),
    ])),
    StructField("carry", StructType([
        StructField("end_location", ArrayType(DoubleType())),
    ])),
    StructField("clearance", StructType([
        StructField("aerial_won", BooleanType()),
        StructField("body_part", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("head", BooleanType()),
        StructField("left_foot", BooleanType()),
        StructField("other", BooleanType()),
        StructField("right_foot", BooleanType()),
    ])),
    StructField("counterpress", BooleanType()),
    StructField("dribble", StructType([
        StructField("nutmeg", BooleanType()),
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("overrun", BooleanType()),
    ])),
    StructField("duel", StructType([
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("type", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("duration", DoubleType()),
    StructField("foul_committed", StructType([
        StructField("advantage", BooleanType()),
        StructField("card", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("offensive", BooleanType()),
        StructField("type", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("foul_won", StructType([
        StructField("advantage", BooleanType()),
        StructField("defensive", BooleanType()),
    ])),
    StructField("goalkeeper", StructType([
        StructField("body_part", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("end_location", ArrayType(DoubleType())),
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("position", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("technique", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("type", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("id", StringType()),
    StructField("index", LongType()),
    StructField("interception", StructType([
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("location", ArrayType(DoubleType())),
    StructField("minute", LongType()),
    StructField("off_camera", BooleanType()),
    StructField("out", BooleanType()),
    StructField("pass", StructType([
        StructField("aerial_won", BooleanType()),
        StructField("angle", DoubleType()),
        StructField("assisted_shot_id", StringType()),
        StructField("body_part", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("cross", BooleanType()),
        StructField("cut_back", BooleanType()),
        StructField("deflected", BooleanType()),
        StructField("end_location", ArrayType(DoubleType())),
        StructField("goal_assist", BooleanType()),
        StructField("height", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("inswinging", BooleanType()),
        StructField("length", DoubleType()),
        StructField("miscommunication", BooleanType()),
        StructField("no_touch", BooleanType()),
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("outswinging", BooleanType()),
        StructField("recipient", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("shot_assist", BooleanType()),
        StructField("switch", BooleanType()),
        StructField("technique", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("through_ball", BooleanType()),
        StructField("type", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("period", LongType()),
    StructField("play_pattern", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])),
    StructField("player", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])),
    StructField("position", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])),
    StructField("possession", LongType()),
    StructField("possession_team", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])),
    StructField("related_events", ArrayType(StringType())),
    StructField("second", LongType()),
    StructField("shot", StructType([
        StructField("aerial_won", BooleanType()),
        StructField("body_part", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("end_location", ArrayType(DoubleType())),
        StructField("first_time", BooleanType()),
        StructField("freeze_frame", ArrayType(StructType([
            StructField("location", ArrayType(DoubleType())),
            StructField("player", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("position", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("teammate", BooleanType()),
        ]))),
        StructField("key_pass_id", StringType()),
        StructField("one_on_one", BooleanType()),
        StructField("open_goal", BooleanType()),
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("statsbomb_xg", DoubleType()),
        StructField("technique", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("type", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("substitution", StructType([
        StructField("outcome", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("replacement", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])),
    StructField("tactics", StructType([
        StructField("formation", LongType()),
        StructField("lineup", ArrayType(StructType([
            StructField("jersey_number", LongType()),
            StructField("player", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("position", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
        ]))),
    ])),
    StructField("team", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])),
    StructField("timestamp", StringType()),
    StructField("type", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])),
    StructField("under_pressure", BooleanType()),
])
events_df = spark.read.schema(events_schema).json(STATSBOMB_PATHS["EVENTS"], multiLine=True, recursiveFileLookup=True)
events_df = events_df.withColumn("source_file", input_file_name())
events_df = events_df.withColumn("loaded_at", current_timestamp())

In [21]:
print(events_df.count())

14874171
